In [40]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import col

In [41]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .getOrCreate()

In [42]:
schema_ratings = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("movie_id", IntegerType(), False),
    StructField("rating", IntegerType(), False),
    StructField("timestamp", IntegerType(), False)
])

We select 80% of users past ratings for train subset adn 20% as test subset.

In [ ]:
ratings = spark.read.option("delimiter", "::").csv("../data/ratings.dat", schema=schema_ratings)
ratings.show()

+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|      1|    1193|     5|978300760|
|      1|     661|     3|978302109|
|      1|     914|     3|978301968|
|      1|    3408|     4|978300275|
|      1|    2355|     5|978824291|
|      1|    1197|     3|978302268|
|      1|    1287|     5|978302039|
|      1|    2804|     5|978300719|
|      1|     594|     4|978302268|
|      1|     919|     4|978301368|
|      1|     595|     5|978824268|
|      1|     938|     4|978301752|
|      1|    2398|     4|978302281|
|      1|    2918|     4|978302124|
|      1|    1035|     5|978301753|
|      1|    2791|     4|978302188|
|      1|    2687|     3|978824268|
|      1|    2018|     4|978301777|
|      1|    3105|     5|978301713|
|      1|    2797|     4|978302039|
+-------+--------+------+---------+
only showing top 20 rows


In [44]:
w = Window.partitionBy("user_id").orderBy(col("timestamp").desc())
ranked = ratings.withColumn("rn", row_number().over(w))
ranked.show()

+-------+--------+------+---------+---+
|user_id|movie_id|rating|timestamp| rn|
+-------+--------+------+---------+---+
|     28|    2132|     5|978985335|  1|
|     28|       1|     3|978985309|  2|
|     28|     912|     5|978985294|  3|
|     28|     527|     5|978985280|  4|
|     28|     916|     4|978985238|  5|
|     28|    2033|     2|978982338|  6|
|     28|     648|     3|978982323|  7|
|     28|     266|     2|978982323|  8|
|     28|     144|     2|978982281|  9|
|     28|     480|     3|978982261| 10|
|     28|    3793|     4|978982233| 11|
|     28|    2657|     4|978982233| 12|
|     28|      47|     2|978982203| 13|
|     28|    1249|     2|978982185| 14|
|     28|     551|     4|978982185| 15|
|     28|    2858|     4|978982159| 16|
|     28|    1266|     3|978126545| 17|
|     28|    1660|     4|978126545| 18|
|     28|    2174|     3|978126491| 19|
|     28|     906|     4|978126477| 20|
+-------+--------+------+---------+---+
only showing top 20 rows


In [45]:
counts = ratings.groupBy("user_id").count()
counts.show()

+-------+-----+
|user_id|count|
+-------+-----+
|    148|  624|
|    463|  123|
|    471|  105|
|    496|  119|
|    833|   21|
|   1088| 1176|
|    243|   33|
|    392|  487|
|    540|   39|
|    623|  172|
|    737|  217|
|    858|  190|
|    897|  203|
|   1025|   33|
|   1084|  180|
|     31|  119|
|    516|  294|
|     85|   39|
|    137|  201|
|    251|   73|
+-------+-----+
only showing top 20 rows


In [46]:
joined = ranked.join(counts, "user_id")

train = joined.filter(col("rn") > col("count") * 0.2).drop("rn", "count")
test  = joined.filter(col("rn") <= col("count") * 0.2).drop("rn", "count")

Save in the same format we read for convenience.

In [ ]:
from pyspark.sql.functions import concat_ws, col

train_out = train.select(
    concat_ws(
        "::",
        col("user_id"),
        col("movie_id"),
        col("rating"),
        col("timestamp")
    ).alias("value")
)

test_out = test.select(
    concat_ws(
        "::",
        col("user_id"),
        col("movie_id"),
        col("rating"),
        col("timestamp")
    ).alias("value")
)

train_out \
    .write \
    .mode("overwrite") \
    .option("header", "false") \
    .csv("../data/ratings_train.dat")

test_out \
    .write \
    .mode("overwrite") \
    .option("header", "false") \
    .csv("../data/ratings_test.dat")